# Framework for Project 1

In [1]:
!pip install qiskit

In [9]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
sim = AerSimulator.from_backend(backend)   # noise from real calibration data
lengths = [0, 20, 40, 80, 160, 320]        # all even
shots = 4000

def error_per_gate(gate, qubit=0):
    circs = []
    for n in lengths:
        qc = QuantumCircuit(1, 1)
        for _ in range(n):
            getattr(qc, gate)(0)
            qc.barrier()
        qc.measure(0, 0)
        circs.append(transpile(qc, backend, optimization_level=0,
                               initial_layout=[qubit]))
    result = sim.run(circs, shots=shots).result()
    p0 = np.array([result.get_counts(i).get('0', 0) / shots
                   for i in range(len(lengths))])
    y = 2 * p0 - 1                          # decays as A * d^n
    ok = y > 0
    slope = np.polyfit(np.array(lengths)[ok], np.log(y[ok]), 1)[0]
    return 1 - np.exp(slope)                # error per gate

count = 0
for q in range(100):                   # repeat on several qubits
    h, x = error_per_gate('h', q), error_per_gate('x', q)
    if h>x:
        count +=1
print(f"H error was greater than X with proportion {count/100}")

/var/folders/kp/mcq688xx02733ht719zs2bbc0000gn/T/ipykernel_92562/24281120.py:26: RankWarning: Polyfit may be poorly conditioned
  slope = np.polyfit(np.array(lengths)[ok], np.log(y[ok]), 1)[0]


H error was greater than X with proportion 0.49
